# Volatility Dynamics 10d — wings priced from the matching volatility regime × time to expiry

**VD 028** (proposed in VD 024 §5). 10c priced every wing quote from one pooled two-year outcome distribution and found the bid CONSISTENT. That pool mixes regimes: standardized tails are about 10× fatter in calm volatility than in volatile volatility.

10d prices each quote from its own **(own-asset volatility tercile × TTE bucket)** cell of the two-year pool. The pool, the consistency gate and the Module 10 clock are the same as in 10c.

**Two pre-registered decisions** (Holm, family α = 0.05, one-sided; joint bootstrap of quote sessions × history dates, B = 1000):

| Decision | Sample | Price | Rejection means |
|---|---|---|---|
| D_long | low tercile (calm) | ask + fee | `CHEAP_AT_ASK`: a long-wing edge |
| D_short | high tercile (volatile) | bid − fee | `RICH_AT_BID`: a short-wing edge |

**Consequence stated in advance:** a rejection opens a defined-risk backtest pre-registration for that side only. No rejection closes the wing line.

Environment: `VD10D_HISTORY_DIR`, `VD10D_ASSETS`, `VD10D_N_BOOT`.

In [1]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 300)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_jump_option_wings as ow
from data_engineering.modeling import volatility_wings_empirical as we
from data_engineering.modeling import volatility_wings_longhistory as wl
from data_engineering.modeling import volatility_wings_regime as wr
ASSETS = [x.strip().upper() for x in os.getenv('VD10D_ASSETS', 'AVAX,BTC,ETH,HYPE,SOL,TRX,XRP').split(',') if x.strip()]
HIST_DIR = Path(os.getenv('VD10D_HISTORY_DIR', str(DATA_ROOT / 'history' / 'price_5m')))
N_BOOT = int(os.getenv('VD10D_N_BOOT', str(wr.N_BOOT)))
VD = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics'
OUT = VD / '10d'
OUT.mkdir(parents=True, exist_ok=True)
print('Module 10d', wr.VERSION, wr.selftest(), '| 10c', wl.VERSION)
print(json.dumps(wr.registration(), indent=2))
manifest = HIST_DIR / 'manifest.json'
if manifest.exists():
    print(manifest.read_text())


Module 10d 0.1.0 {'version': '0.1.0', 'ok': True} | 10c 0.1.0
{
  "version": "0.1.0",
  "n_boot": 1000,
  "seed": 1028,
  "alpha_holm": 0.05,
  "decisions": {
    "D_long": "low tercile, ask net of fee, CHEAP if Holm-rejected H0 M>=0",
    "D_short": "high tercile, bid net of fee, RICH if Holm-rejected H0 M<=0"
  },
  "cell": "(own-asset v tercile from 2-year pool cutoffs) x (Module 10 TTE bucket)"
}
{
  "AVAX": {
    "status": "ok",
    "source": [
      "binance_spot"
    ],
    "symbol": [
      "AVAXUSDT"
    ],
    "first": "2024-09-01 00:05:00+00:00",
    "last": "2026-10-11 00:05:00+00:00",
    "n_bars": 221761
  },
  "BTC": {
    "status": "ok",
    "source": [
      "binance_spot"
    ],
    "symbol": [
      "BTCUSDT"
    ],
    "first": "2024-09-01 00:05:00+00:00",
    "last": "2026-10-11 00:05:00+00:00",
    "n_bars": 221761
  },
  "ETH": {
    "status": "ok",
    "source": [
      "binance_spot"
    ],
    "symbol": [
      "ETHUSDT"
    ],
    "first": "2024-09-01 00:05:0

## 1. Two-year pool and consistency gate (as 10c)

In [2]:
t0 = time.time()
hist = wl.load_history(HIST_DIR, ASSETS)
print(hist.groupby('underlying').agg(first=('state_time', 'min'), last=('state_time', 'max'), n=('close', 'size'),
                                     source=('source', lambda s: ','.join(sorted(set(map(str, s)))))))
frame = wl.return_frame(hist)
from data_engineering.modeling import volatility_vd_b_core as vb
con = duckdb.connect(str(DB_PATH), read_only=True)
dbf = con.execute("""SELECT state_time, underlying, return_5m FROM time_series_feature_state
                     WHERE return_5m IS NOT NULL""").df()
con.close()
dbf = dbf[dbf.underlying.isin(ASSETS) & (pd.to_datetime(dbf.state_time, utc=True) < ow.VRP_CONFIRMATION_START)]
clock = wl.module10_clock(vb.build_grid(dbf))
oc = wl.long_outcomes(frame, clock=clock)
print(f'{len(oc):,} long-history origins in {time.time()-t0:.0f}s')
q = pd.read_csv(VD / '10' / '10_priced_quotes.csv.gz')
o10b = pd.read_csv(VD / '10b' / '10b_origin_outcomes.csv.gz')
pool_all, overlap = wl.split(oc, sorted(q.session.unique()))
gate = wl.consistency_gate(overlap, o10b)
display(gate.round(3))
failed = gate.loc[~gate['pass'], 'underlying'].tolist()
gate_fail = len(failed) > wl.GATE_MAX_FAILS
pool = pool_all[~pool_all.underlying.isin(failed)].reset_index(drop=True)
print('gate failures:', failed, '| GATE_FAIL:', gate_fail)
print(f'pool: {len(pool):,} origins, {pool.session.nunique()} dates, {pool.session.min()} -> {pool.session.max()}')


                               first                      last       n        source
underlying                                                                          
AVAX       2024-09-01 00:05:00+00:00 2026-10-11 00:05:00+00:00  221761  binance_spot
BTC        2024-09-01 00:05:00+00:00 2026-10-11 00:05:00+00:00  221761  binance_spot
ETH        2024-09-01 00:05:00+00:00 2026-10-11 00:05:00+00:00  221761  binance_spot
HYPE       2026-09-24 11:05:00+00:00 2026-10-11 00:05:00+00:00    4765  binance_spot
SOL        2024-09-01 00:05:00+00:00 2026-10-11 00:05:00+00:00  221761  binance_spot
TRX        2024-09-01 00:05:00+00:00 2026-10-11 00:05:00+00:00  221761  binance_spot
XRP        2024-09-01 00:05:00+00:00 2026-10-11 00:05:00+00:00  221761  binance_spot
1,060,459 long-history origins in 139s


,underlying,n_matched,corr,sd_ratio,pass
0,AVAX,3998,0.999,0.999,True
1,BTC,3998,0.997,1.026,True
2,ETH,3998,0.997,0.963,True
3,HYPE,3314,0.999,0.979,True
4,SOL,3998,0.999,0.984,True
5,TRX,3998,0.974,1.007,True
6,XRP,3998,0.998,0.979,True


gate failures: [] | GATE_FAIL: False
pool: 1,027,968 origins, 749 dates, 2024-09-01 -> 2026-09-19


## 2. Volatility terciles for pool and quotes

In [3]:
cuts = wr.cutoffs(pool)
pool['tercile'] = wr.classify(pool, cuts)
excluded = json.loads((VD / '10' / '10_decision.json').read_text()).get('excluded_assets_tick_artifact', [])
prim = ow.primary_sample(q, excluded).reset_index(drop=True)
prim = wr.attach_quote_v(prim, overlap)
prim = prim[np.isfinite(prim.v) & ~prim.underlying.isin(failed)].reset_index(drop=True)
prim['tercile'] = wr.classify(prim, cuts)
prim = wr.net_prices(prim)
display(pd.DataFrame(cuts, index=['cut_1/3', 'cut_2/3']).T)
display(prim.groupby(['tercile', 'tte_bucket']).size().unstack(fill_value=0))
display(pool.groupby(['tercile', 'tte_bucket']).size().unstack(fill_value=0))

,cut_1/3,cut_2/3
AVAX,2.818708e-06,5.934083e-06
BTC,6.920803e-07,1.633304e-06
ETH,1.531489e-06,3.554473e-06
SOL,2.270503e-06,4.974251e-06
TRX,2.814067e-07,7.049433e-07
XRP,1.648994e-06,4.441434e-06


tte_bucket,"(120.0, 240.0]","(240.0, 480.0]","(480.0, 960.0]","(60.0, 120.0]","(960.0, 1200.0]"
tercile,,,,,
high,24,196,1158,2,807
low,86,775,2881,8,2625
mid,158,330,1446,26,1831
na,100,343,1010,31,648


tte_bucket,"(120.0, 240.0]","(240.0, 480.0]","(480.0, 960.0]","(60.0, 120.0]","(960.0, 1200.0]"
tercile,,,,,
high,33416,85298,159916,13320,49601
low,37577,58514,130072,22442,92167
mid,36857,71612,140860,18166,73656


## 3. Decisions

In [4]:
t0 = time.time()
dec = wr.decide(prim, pool, n_boot=N_BOOT)
print(f'joint bootstrap ({time.time()-t0:.0f}s)')
print(json.dumps(dec, indent=2, default=float))

joint bootstrap (261s)
{
  "n_low": 6375,
  "n_high": 2187,
  "sessions_low": 19,
  "sessions_high": 15,
  "M_ask_low": {
    "M": 0.5368934816450495,
    "ci_low": 0.22917022729491385,
    "ci_high": 1.2117875088842613,
    "p_one_sided": 0.999,
    "holm_rejected": false
  },
  "M_bid_high": {
    "M": -0.231979684861501,
    "ci_low": -1.0351333803460097,
    "ci_high": 0.26320980790463666,
    "p_one_sided": 0.835,
    "holm_rejected": false
  },
  "D_long_verdict": "NOT_CHEAP",
  "D_short_verdict": "NOT_RICH"
}


## 4. Reported: every cell at bid (net), mid and ask (net)

In [5]:
cells = wr.cell_table(prim, pool, n_boot=500)
display(cells.pivot_table(index=['tercile', 'tte_bucket'], columns='price', values='M').round(3))

price                    ask_net_usd  bid_net_usd  mid_usd
tercile tte_bucket                                        
high    (120.0, 240.0]         1.271       -0.135    0.788
        (240.0, 480.0]         2.877        0.119    2.173
        (480.0, 960.0]         1.268       -0.399    0.643
        (60.0, 120.0]          1.821       -1.332    1.064
        (960.0, 1200.0]        0.590       -0.640    0.135
low     (120.0, 240.0]         1.531        0.542    1.102
        (240.0, 480.0]         1.947        1.049    1.560
        (480.0, 960.0]         0.661       -0.237    0.302
        (60.0, 120.0]          1.754        0.372    1.230
        (960.0, 1200.0]        0.226       -0.502   -0.130
mid     (120.0, 240.0]         1.798        0.562    1.333
        (240.0, 480.0]         1.143        0.035    0.710
        (480.0, 960.0]         0.671       -0.518    0.318
        (60.0, 120.0]          1.623        0.164    1.100
        (960.0, 1200.0]        0.238       -0.746   -0.145

## 5. Outputs

In [6]:
gate.to_csv(OUT / '10d_consistency_gate.csv', index=False)
cells.to_csv(OUT / '10d_cell_table.csv', index=False, float_format='%.6g')
pd.DataFrame(cuts, index=['cut_1', 'cut_2']).T.to_csv(OUT / '10d_tercile_cutoffs.csv')
(OUT / '10d_decision.json').write_text(json.dumps({'decision': dec, 'gate_failures': failed, 'gate_fail': gate_fail,
                                                    'registration': wr.registration()}, indent=2, default=float))
meta = {'version': wr.VERSION, 'module': '10d_regime_conditioned_wings', 'history_dir': str(HIST_DIR), 'n_boot': N_BOOT,
        'n_quotes': int(len(prim)), 'n_pool_origins': int(len(pool)), 'pool_last_session': wl.POOL_LAST_SESSION,
        'consumes': ['10/10_priced_quotes.csv.gz', '10b/10b_origin_outcomes.csv.gz', '10/10_decision.json']}
(OUT / '10d_run_metadata.json').write_text(json.dumps(meta, indent=2, default=float))
print(json.dumps(meta, indent=2, default=float))

{
  "version": "0.1.0",
  "module": "10d_regime_conditioned_wings",
  "history_dir": "D:\\MarketData\\Deribit\\history\\price_5m",
  "n_boot": 1000,
  "n_quotes": 14493,
  "n_pool_origins": 1027968,
  "pool_last_session": "2026-09-19",
  "consumes": [
    "10/10_priced_quotes.csv.gz",
    "10b/10b_origin_outcomes.csv.gz",
    "10/10_decision.json"
  ]
}
